##PARAMETERS

In [0]:
#Catalog name
catalog = "workspace"

#Source Schema
source_schema = "silver"

#Source Object
source_object = "silver_bookings"

#CDC Column
cdc_col = "modifyDate"

#Back Dated Refresh
backdated_refresh = ""

#Source Fact Table
fact_table = f"{catalog}.{source_schema}.{source_object}"

#TARGET Schema
target_schema = "gold"

#Target Object
target_object = "FactBookings"

#Fact Key Cols List
fact_key_cols = ["DimPassengersKey", "DimFlightsKey", "DimAirportsKey", "booking_date"]

In [0]:
dimensions = [{
    "table": f"{catalog}.{target_schema}.DimPassengers",
    "alias": "DimPassengers",
    "join_keys": [("passenger_id", "passenger_id")] #(fact_col, dim_col)
    },
    {
    "table": f"{catalog}.{target_schema}.DimFlights",
    "alias": "DimFlights",
    "join_keys": [("flight_id", "flight_id")] #(fact_col, dim_col)
    },
    {
    "table": f"{catalog}.{target_schema}.DimAirports",
    "alias": "DimAirports",
    "join_keys": [("airport_id", "airport_id")] #(fact_col, dim_col)
    },
]
    
#Columns you want to keep from Fact table (besides the surrogate keys) (as in fact table we only keep numerical columns and the surrogate keys column)
fact_columns = ["amount", "booking_date", "modifyDate"]

**Last Load Date**

In [0]:
#No Back dated Refresh
if len(backdated_refresh) == 0:
    
    #if table exists in the destination
    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

        last_load = spark.sql(f"SELECT MAX({cdc_col}) FROM {catalog}.{target_schema}.{target_object}").collect()[0][0]
    else:
        last_load = "1900-01-01 00:00:00"  
#Yes Back Dates Refresh
else:
    last_load = backdated_refresh
#test the last load
last_load

datetime.datetime(2025, 12, 23, 0, 10, 6, 681000)

##DYNAMIC FACT QUERY [Bring Keys]

In [0]:
def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_col, processing_date):
    fact_alias = "f"

    #Base columns to select
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    #Build joins dynamically
    join_clauses = []
    for dim in dimensions:
        table_full = dim["table"]
        alias = dim["alias"]
        table_name = table_full.split('.')[-1]
        surrogate_key = f"{alias}.{table_name}Key"
        select_cols.append(surrogate_key) 

        #BUILD ON CLAUSE
        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{dk}" for (fk, dk) in dim["join_keys"] 
        ]
        join_clause = f"LEFT JOIN {table_full} {alias} ON " + " AND ".join(on_conditions)
        join_clauses.append(join_clause)

    #Final SELECT and JOIN clauses
    select_clause = ",\n     ".join(select_cols)
    joins = "\n".join(join_clauses)

    #WHERE CLAUSE FOR INCREMENTAL FILTERING
    where_clause = f"{fact_alias}.{cdc_col} >= DATE('{last_load}')"

    #Final query
    query = f"""
SELECT {select_clause}
FROM {fact_table} {fact_alias}
{joins}
WHERE {where_clause}
""".strip()
    return query

In [0]:
query = generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_col, last_load)
print(query)

SELECT f.amount,
     f.booking_date,
     f.modifyDate,
     DimPassengers.DimPassengersKey,
     DimFlights.DimFlightsKey,
     DimAirports.DimAirportsKey
FROM workspace.silver.silver_bookings f
LEFT JOIN workspace.gold.DimPassengers DimPassengers ON f.passenger_id = DimPassengers.passenger_id
LEFT JOIN workspace.gold.DimFlights DimFlights ON f.flight_id = DimFlights.flight_id
LEFT JOIN workspace.gold.DimAirports DimAirports ON f.airport_id = DimAirports.airport_id
WHERE f.modifyDate >= DATE('2025-12-23 00:10:06.681000')


### **DF_FACT

In [0]:
df_fact = spark.sql(query)

In [0]:
df_fact.display()

amount,booking_date,modifyDate,DimPassengersKey,DimFlightsKey,DimAirportsKey
850.72,2025-05-29,2025-12-23T00:10:06.681Z,47,12,48
376.63,2025-06-09,2025-12-23T00:10:06.681Z,11,109,3
534.02,2025-06-03,2025-12-23T00:10:06.681Z,77,21,12
1333.7,2025-06-16,2025-12-23T00:10:06.681Z,66,1,39
1334.96,2025-06-17,2025-12-23T00:10:06.681Z,180,17,8
296.13,2025-05-18,2025-12-23T00:10:06.681Z,68,106,4
460.14,2025-04-05,2025-12-23T00:10:06.681Z,185,63,16
1402.02,2025-06-04,2025-12-23T00:10:06.681Z,172,44,17
1444.51,2025-05-16,2025-12-23T00:10:06.681Z,94,76,16
292.39,2025-05-16,2025-12-23T00:10:06.681Z,126,84,34


##**UPSERT**

In [0]:
#Fact Key Column Merge Condition

fact_key_cols_str = " AND ".join([f"src.{col} = trg.{col}" for col in fact_key_cols])
fact_key_cols_str

'src.DimPassengersKey = trg.DimPassengersKey AND src.DimFlightsKey = trg.DimFlightsKey AND src.DimAirportsKey = trg.DimAirportsKey AND src.booking_date = trg.booking_date'

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
  dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
  dlt_obj.alias("trg").merge(df_fact.alias("src"), fact_key_cols_str)\
    .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
    .whenNotMatchedInsertAll()\
    .execute()


else:
  df_fact.write.format("delta").mode("append")\
    .saveAsTable(f"{catalog}.{target_schema}.{target_object}")



In [0]:
%sql
SELECT * FROM workspace.gold.factbookings

amount,booking_date,modifyDate,DimPassengersKey,DimFlightsKey,DimAirportsKey
850.72,2025-05-29,2025-12-23T00:10:06.681Z,47,12,48
376.63,2025-06-09,2025-12-23T00:10:06.681Z,11,109,3
534.02,2025-06-03,2025-12-23T00:10:06.681Z,77,21,12
1333.7,2025-06-16,2025-12-23T00:10:06.681Z,66,1,39
1334.96,2025-06-17,2025-12-23T00:10:06.681Z,180,17,8
296.13,2025-05-18,2025-12-23T00:10:06.681Z,68,106,4
460.14,2025-04-05,2025-12-23T00:10:06.681Z,185,63,16
1402.02,2025-06-04,2025-12-23T00:10:06.681Z,172,44,17
1444.51,2025-05-16,2025-12-23T00:10:06.681Z,94,76,16
292.39,2025-05-16,2025-12-23T00:10:06.681Z,126,84,34
